<a href="https://colab.research.google.com/github/FranciscoDegiovani/tcga-brca-multiomics-survival/blob/main/notebooks/mofa2_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Install mofapy2

Training MOFA2 directly in Python here — ran into a basilisk/pyenv installation issue trying to run it from R on Windows, so switching engines for this step while keeping the R-based preprocessing.

In [1]:
!pip install mofapy2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 7.5 MB/s eta 0:00:00


## Load the matrices exported from R

RNA-seq (log2 CPM, top 5000 variable genes) and DNA methylation (beta values, top 5000 variable probes), both already matched and ordered to the same 866 samples in the R preprocessing step.

In [2]:
import pandas as pd
import numpy as np

rna_df = pd.read_csv('rna_filtered_samples_by_features.csv', index_col=0)
meth_df = pd.read_csv('meth_filtered_samples_by_features.csv', index_col=0)

print(f"RNA: {rna_df.shape}")
print(f"Methylation: {meth_df.shape}")

# Confirm sample order matches between the two views
print(f"Samples match: {(rna_df.index == meth_df.index).all()}")

RNA: (866, 5000)
Methylation: (866, 5000)
Samples match: True


## Set up the MOFA2 model

Configuring the two omics layers as separate "views" and setting training parameters — 10 factors as a starting point, consistent with the R setup that hit the basilisk error.

In [4]:
# Data must be structured as [view][group] — outer list = views, inner list = groups.
# We have 2 views (RNA, Methylation) and 1 group (all samples together).
data_mat = [[rna_df.values], [meth_df.values]]

ent.set_data_matrix(data_mat, views_names=["RNA", "Methylation"],
                     samples_names=[rna_df.index.tolist()],
                     features_names=[rna_df.columns.tolist(), meth_df.columns.tolist()])

Groups names not provided, using default naming convention:
- group1, group2, ..., groupG

Successfully loaded view='RNA' group='group0' with N=866 samples and D=5000 features...
Successfully loaded view='Methylation' group='group0' with N=866 samples and D=5000 features...




## Build and train the model

In [5]:
ent.set_model_options(factors=10)
ent.set_train_options(seed=42, convergence_mode="fast")

ent.build()
ent.run()

Model options:
- Automatic Relevance Determination prior on the factors: False
- Automatic Relevance Determination prior on the weights: True
- Spike-and-slab prior on the factors: False
- Spike-and-slab prior on the weights: True
Likelihoods:
- View 0 (RNA): gaussian
- View 1 (Methylation): gaussian




######################################
## Training the model with seed 42 ##
######################################


ELBO before training: -64707133.54 

Iteration 1: time=2.02, ELBO=-10227520.38, deltaELBO=54479613.153 (84.19413776%), Factors=10
Iteration 2: time=2.02, ELBO=-9295294.46, deltaELBO=932225.928 (1.44068494%), Factors=10
Iteration 3: time=2.04, ELBO=-9229138.46, deltaELBO=66156.000 (0.10223911%), Factors=10
Iteration 4: time=2.77, ELBO=-9195587.46, deltaELBO=33550.995 (0.05185053%), Factors=10
Iteration 5: time=3.13, ELBO=-9170470.40, deltaELBO=25117.063 (0.03881653%), Factors=10
Iteration 6: time=2.23, ELBO=-9151876.07, deltaELBO=18594.325 (0.02873613%), Factors=10
Itera

In [6]:
outfile = "mofa_model.hdf5"
ent.save(outfile)

from google.colab import files
files.download(outfile)

Saving model in mofa_model.hdf5...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
from google.colab import files
files.download("mofa_model.hdf5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>